# Run and plot a `LAPDSim1D` simulation

This notebook:
1. Builds a `LAPDSim1D` from `default_config()`, applies your **parameter/flag overrides**, runs it, and saves an HDF5 result.
2. Renders the app-style contour and summary plots inline.
3. Shows main-discharge time-slice profiles at **15 ms and 19 ms** only.

All z-axis plots include the vertical dashed **port markers** (ports 21, 29, 41) used by `bapsf_app`.

Run the notebook from `cablp/scripts/` (it imports the CLI helper `plot_sim1d_run.py` from that directory).

In [ ]:
%matplotlib inline

import sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display

from cablp.solvers._sim1d import (
    LAPDSim1D,
    ProgressPrinter1D,
    default_config,
    load_result_hdf5,
    summarize_result,
)

# Resolve the notebook support files when Jupyter starts in either cablp/scripts/
# or the bapsf-transport repository root.
SCRIPT_DIR = Path.cwd()
if not (SCRIPT_DIR / "plot_sim1d_run.py").exists():
    SCRIPT_DIR = Path.cwd() / "cablp/scripts"
if not (SCRIPT_DIR / "plot_sim1d_run.py").exists():
    raise FileNotFoundError("Run this notebook from cablp/scripts or the repository root")

# Importing the CLI plotting helper sets the Agg backend, so re-assert inline.
sys.path.insert(0, str(SCRIPT_DIR))
import plot_sim1d_run as psr
%matplotlib inline

# Self-contained ES1 export from bapsf-lapd-data-analysis.
ES1_OVERLAY_PATH = SCRIPT_DIR / "data/es1_sim1d_overlay.npz"
es1_overlay = np.load(ES1_OVERLAY_PATH, allow_pickle=False)

# Shift all experimental time axes relative to the simulation main-discharge clock.
ES1_TIME_SHIFT_MS = 0.0

# Finite-ion-temperature coefficient in the Isat proxy sqrt(Te + gamma_i*Ti).
# gamma_i=1 is the isothermal-ion choice; the cold-ion gamma_i=0 curve is also shown.
ISAT_GAMMA_I = 1.0
ISAT_ERRORBAR_STRIDE = 5  # 5 exported bins ~= 50 us between displayed error bars
print(f"Loaded ES1 overlay: {ES1_OVERLAY_PATH}")


## Configuration

Edit `param_overrides` / `flag_overrides` to override the defaults. Anything left commented uses the value from `default_config()`. Call `default_config()` in a scratch cell to see every available key.

The run controls below map to `sim.start_simulation(...)`; leave them `None` to use the config defaults.

In [ ]:
# --- Parameter overrides (input_dict keys) ---
# Model state 2026-07-18: ADAS rates, beam excitation, CX-derived momentum
# transfer, decoupled thermalization. Every key is documented in the
# *_defaults() docstrings in solvers/_sim1d/core/config.py; THESIS_NOTES.md
# carries what each choice can and cannot claim.
Ts = 273.15 + 1735

param_overrides = {
    # "gas_type": "He",
    #
    # =========================================================================
    # GEOMETRY / NEUTRAL TRANSPORT (resolved boundary, knudsen)
    # =========================================================================
    # Resolved source/end boundary (BOUNDARY_REGIONS_PLAN.md). knudsen is the
    # only mesh-consistent neutral transport; molecular_flow is retained solely
    # to reproduce historical results. nx=120 is the tuning mesh: port-sampled
    # benchmark metrics move <=3% vs nx=185; PEAK quantities are still not
    # mesh-converged (gate #7) -- don't quote them.
    "neutral_exchange_model": "knudsen",
    "nx": 120,
    # "nx_gap": 5,          # cathode-anode gap cells (10 cm each at 5)
    #
    # =========================================================================
    # PRIMARY TUNING KNOBS: fueling and cathode (stage i/ii levers)
    # =========================================================================
    "V_bank": 180.0,        # measured -- not a free parameter
    "R_comp": 0.010,        # measured -- not a free parameter
    "T_s": Ts,              # hand-tuned: sets emission -> discharge current
    "S_gp": 3700,           # hand-tuned: gas puff [sccm]
    "S_gp_decay_target": 2000,
    "tau_gp_pulse_duration": 1e-3,
    "tau_gp_decay_duration": 5e-3,
    "Rp": 15.0,
    "R_cath": 15.0,
    #
    # =========================================================================
    # ATOMIC RATES (ADAS GCR '96 -- see cablp/vars/adas/README.md)
    # =========================================================================
    # "adas": effective SCD/ACD particle rates (incl. the stepwise/metastable
    # ionization the direct rate misses -- 3-6x at 3-5 eV) and radiation-only
    # PLT/PRB cooling, consistent with the separate ionization-cost term.
    # "janev" restores the historical fits (whose He I cooling fit
    # double-counts the ionization cost against b_Qen=1 -- THESIS_NOTES sec. 2).
    "atomic_rate_model": "adas",
    "b_Qei": 1,             # genuine O(1) sensitivity knobs under adas
    "b_Qen": 1,
    "b_Qcx": 1,
    # Optional Te-shape on b_Qei/b_Qen: multiplies by (Te/b_Q_Te_ref_eV)**exp.
    # "b_Qei_Te_exp": 0.0,
    # "b_Qen_Te_exp": 0.0,
    # "b_Q_Te_ref_eV": 5.0,
    # "b_ioniz": 1.0,       # scales SCD ionization under adas
    # "b_rec_rad": 1.0,     # scales the whole ACD sink under adas
    # NB b_rec_3b is INERT under adas (ACD already includes three-body).
    #
    # =========================================================================
    # CATHODE BEAM
    # =========================================================================
    # Beam-driven neutral excitation: each event radiates ~21 eV as He I light
    # (previously deposited as heat) and shortens the inelastic deposition
    # length. 1.0 = the 2^1P channel alone; 1.4 approximates the full singlet
    # manifold (the 0.4 is an estimate -- THESIS_NOTES item #9). 0 = historical.
    "b_beam_excitation": 1.4,
    #
    # =========================================================================
    # ION-NEUTRAL CLOSURE (THESIS_NOTES gate #2 -- read before tuning these)
    # =========================================================================
    # Constant drag at the calibrated 0.5. This is a STAND-IN for the missing
    # neutral-momentum / radial channel, not validated physics: the physical
    # "slip" entrainment closure predicts ~5x less drag in the dense column and
    # measurably under-confines (n 0.76 -> 0.60, Te 1.37 -> 1.81 on the ES1
    # benchmark). Any density agreement obtained by tuning around this constant
    # must be presented as calibrated compensation.
    "b_ion_neutral_drag": 1,
    "ion_neutral_drag_model": "constant",
    # "ion_neutral_drag_model": "slip",  # physical alternative (set drag b to 1)
    # "b_slip_entrainment": 1.0,         # scales the slip closure's E
    #
    # Thermal equilibration scale, decoupled from the drag scalar (a momentum
    # slip has no business scaling a temperature-relaxation rate).
    "b_ion_neutral_thermalization": 1.0,
    # Momentum-transfer rate from the in-repo CX table (sigma_mt ~ 2*sigma_cx
    # + Langevin floor), consistent with the CX energy channel and carrying the
    # velocity dependence the constant sigma lacks (constant crosses it at
    # ~0.5 eV: too small in the afterglow, too large in the warm column).
    "sigma_in_model": "cx_derived",
    # "sigma_in_cm2": 5.0e-15,  # only used by sigma_in_model="constant"
    #
    # Ion-neutral thermal equilibration relaxes Ti toward Tn_fit, NOT Tn_K.
    # Tn_fit defaults to 0.1 eV = 1160 K (the reaction-rate-fit neutral
    # temperature) while Tn_K is 300 K = 0.026 eV. Note 0.1 eV is also exactly
    # Ti_floor, so the equilibration drives Ti onto the floor rather than to
    # the gas temperature. A target below Ti_floor will fight the floor.
    # "Tn_fit": 0.1,
    #
    # =========================================================================
    # RESOLVED-BOUNDARY KNOBS (sensitivity ranking: THESIS_NOTES section 3)
    # =========================================================================
    # "eta": 0.358,                      # anode opacity: -56% thermal at 0.6
    # "S_pump_L": 2000.0,                # [L/s]; x2 -> +15% thermal
    # "S_pump_R": 4000.0,
    # "pump_elbow_conductance_lps": None,  # 2000 -> -22% thermal
    # "Lcs": 0.0, "Rcs": 0.0,            # annular duct: +40% peak Te, ~0 thermal
    # "Rsup": 0.0,                       # support rods: negligible
    # "b_presheath_length": 1.0,         # 0 = historical constant sheath factor
    #
    # =========================================================================
    # TIME INTEGRATION (operator-split path only)
    # =========================================================================
    # The three keys below are what make the split step second-order, and they
    # are three SEPARATE first-order error terms: leaving any one at its default
    # caps the whole step at 1st order no matter what the other two say. Measured
    # order, scripts/verify_sim1d_order.py at 62 cells:
    #
    #   picard  splitting   backward_euler  crank_nicolson  tr_bdf2
    #     0       lie            0.97            1.01         1.02
    #     4       lie            0.97            1.01         1.02
    #     0       strang         0.98            1.04         0.98
    #     4       strang         0.99            1.99         2.00
    #
    # backward_euler staying at ~1.0 everywhere is the negative control: it
    # cannot be 2nd order at any dt. Full package costs ~1.3-1.45x wall clock;
    # the extra tridiagonal solves are cheap next to the reaction-rate
    # evaluations, which run the same number of times either way.
    #
    # NB a production discharge will NOT show 2nd order: the floors bind on ~42%
    # of cell-visits and phase transitions are threshold-triggered, so the step
    # degrades to 1st order wherever those engage. See NUMERICS.md.
    #
    # Substep discretization:
    #   "backward_euler"  theta=1    over-diffusive but unconditionally monotone,
    #                                so it cannot undershoot the floors
    #   "shifted"         theta=0.6  ~1/5 of backward Euler's error constant;
    #                                damps stiff ringing ~2/3 per step
    #   "crank_nicolson"  theta=0.5  2nd-order, but rings at undamped amplitude
    #                                on stiff modes (not L-stable)
    #   "tr_bdf2"                    2nd-order AND L-stable (trapezoidal stage
    #                                then BDF2): ~60x less ringing than CN and
    #                                half its error, for 2 solves not 1
    # Audited on this config: crank_nicolson clips the Te floor once in ~20k
    # solves (at plasma launch, ~1e-10 of the column thermal energy); tr_bdf2
    # and backward_euler never clip.
    "implicit_heat_scheme": "tr_bdf2",
    # Operator composition. "lie" = A(dt) then B(dt), 1st order. "strang" =
    # B(dt/2), A(dt), B(dt/2), cancelling the leading dt*[A,B] commutator term.
    # Costs one extra heat substep, not one extra explicit step.
    "operator_splitting": "strang",
    # Conductivity evaluation. 0 freezes Braginskii kappa ~ T^2.5 at the step
    # start, which is 1st order however good the substep scheme is. 1 iteration
    # already reaches 2nd order; 2 buys fixed-point margin at large dt for
    # ~0.15x more wall clock. Not worth enabling for backward_euler/shifted --
    # it cannot help their order and slightly worsens their error constant.
    "heat_picard_iterations": 2,
    "heat_picard_tol": 1e-10,
}

# --- Flag overrides (input_flags keys) ---
flag_overrides = {
    # Resolved machine end: plenum / cathode-anode gap / anode mesh / collector,
    # with machine coordinates (cathode surface at z=0, plenum at negative z).
    # Resolved results carry a few-percent gap-mesh uncertainty (gate #5) and
    # peak quantities are not converged (gate #7).
    "resolved_boundaries": True,
    # "implicit_heat_conduction": True,
    # "ion_neutral_drag": True,
    "ion_neutral_drag_cx_only": False,
    # Elastic ion-neutral thermal equilibration: relaxes Ti toward Tn_fit at the
    # elastic collision rate, the elastic companion to the CX cooling Q_cx.
    # Scaled by b_ion_neutral_thermalization above (decoupled from the drag
    # scalar as of 2026-07-18); vanishes if ion_neutral_drag_cx_only is True,
    # since then there is no elastic fraction.
    "ion_neutral_thermalization": True,
    # "icool_recomb": False,  # PRB recombination radiation under adas; negligible
}

# --- Run controls (None => config default) ---
t_end = None            # [s] final time
dt = None               # [s] fixed step; None => adaptive
operator_split = None   # None => use implicit_heat_conduction flag
max_steps = None        # accepted-step cap; 0 => unlimited

output_path = "sim1d_run.h5"

## Run the simulation

In [ ]:
params, flags = default_config()
params.update(param_overrides)
flags.update(flag_overrides)

sim = LAPDSim1D(params, flags)
sim.start_simulation(
    t_end=t_end,
    dt=dt,
    operator_split=operator_split,
    max_steps=max_steps,
    progress_tracker=ProgressPrinter1D(),
)

out = Path(output_path)
out.parent.mkdir(parents=True, exist_ok=True)
sim.save_result(out, sim.get_results(), params=params, flags=flags)

# Reload from HDF5 so `result` carries params/flags and sim3-compatible aliases.
# The in-memory get_results() result has no .params, which otherwise shows up as
# NaN (V_bank, S_gp, ...) in the plot titles. This matches how the CLI plot
# script consumes results.
result = load_result_hdf5(out)

s = summarize_result(result)
print(f"steps={result.steps}, final_time={result.final_time:.4e} s, saves={len(result.time)}, output={out}")
print(
    f"finite={s.finite}, "
    f"n=[{s.n_min:.3e}, {s.n_max:.3e}] cm^-3, "
    f"Te=[{s.Te_min:.3e}, {s.Te_max:.3e}] eV, "
    f"Ti=[{s.Ti_min:.3e}, {s.Ti_max:.3e}] eV"
)

## Plots

Port markers are the vertical gray lines used by `bapsf_app`, at the LAPD probe ports (sim convention, z=0 at the source end):

| port | z [cm] | style |
|------|--------|-------|
| 11   | 470    | dotted |
| 21   | 790    | dashed |
| 29   | 1045   | dashed |
| 41   | 1429   | dashed |
| 50   | 1717   | dotted |

Edit `PORT_Z_SIM_CM` (dashed) / `PORT_Z_DOTTED_CM` (dotted) if your geometry differs. The Cathode panel also overlays the ES1 mean discharge current and the sign-inverted cathode-anode voltage. Shaded bands are SEM across the 16 stored ES1 traces. Adjust `ES1_TIME_SHIFT_MS` above if the SIS trigger and simulation main-discharge clocks need alignment.


In [ ]:
# Port probe positions [cm], sim convention (z=0 at source end).
# LAPD ports are ~31.95 cm apart; z ~= 31.95*port + 119 cm. Ports 21/41 are the
# probe ports (bapsf_app's hardcoded 20/40 markers were interferometers).
PORT_Z_SIM_CM = {21: 790.0, 29: 1045.0, 41: 1429.0}   # dashed markers
PORT_Z_DOTTED_CM = {11: 470.0, 50: 1717.0}            # dotted markers
ALL_PORT_Z_CM = {**PORT_Z_DOTTED_CM, **PORT_Z_SIM_CM}


def add_port_lines(ax):
    """Draw the bapsf_app vertical port markers on a z-axis plot."""
    for z in PORT_Z_SIM_CM.values():
        ax.axvline(z, color="gray", lw=0.8, ls="--", alpha=0.7)
    for z in PORT_Z_DOTTED_CM.values():
        ax.axvline(z, color="gray", lw=0.8, ls=":", alpha=0.7)


# Reproduce the CLI time axis (t=0 at main discharge, milliseconds).
time_origin = psr._time_origin(result, "main_discharge")
time_scale, time_label = psr._time_unit("ms")
shifted_s = np.asarray(result.time, dtype=float) - time_origin
t_plot = shifted_s * time_scale
t_slice_ms = shifted_s * 1.0e3
z_cm = np.asarray(result.z_cm, dtype=float)
phase_events = psr._shifted_phase_events(result, time_origin, time_scale)

In [ ]:
figures = {
    "Summary": psr._plot_summary(result, t_plot, time_label, phase_events),
    "Densities": psr._plot_densities(result, z_cm, t_plot, time_label, phase_events),
    "Temperatures": psr._plot_temperatures(result, z_cm, t_plot, time_label, phase_events),
    "Velocity": psr._plot_velocity(result, z_cm, t_plot, time_label, phase_events),
    "Energy terms": psr._plot_energy_terms(result, t_plot, time_label, phase_events),
    "Cathode": psr._plot_cathode(result, t_plot, time_label, phase_events),
    "Phase": psr._plot_phase(result, t_plot, time_label, phase_events),
}

# Overlay ES1 cathode diagnostics on the existing current/beam-voltage axes.
cathode_fig = figures["Cathode"]
if cathode_fig is not None:
    current_ax = next(
        (ax for ax in cathode_fig.axes if ax.get_ylabel() == "I_tot [A]"),
        None,
    )
    voltage_ax = next(
        (ax for ax in cathode_fig.axes if ax.get_ylabel() == "V_b [V]"),
        None,
    )
    if current_ax is None or voltage_ax is None:
        raise RuntimeError("Could not identify the Cathode current/voltage axes")

    exp_time_ms = (
        np.asarray(es1_overlay["discharge_time_ms"], dtype=float)
        + ES1_TIME_SHIFT_MS
    )
    visible = (exp_time_ms >= np.nanmin(t_plot)) & (exp_time_ms <= np.nanmax(t_plot))
    exp_current = np.asarray(es1_overlay["discharge_current_mean_a"], dtype=float)
    exp_current_sem = np.asarray(es1_overlay["discharge_current_sem_a"], dtype=float)
    exp_voltage = np.asarray(
        es1_overlay["discharge_voltage_positive_mean_v"],
        dtype=float,
    )
    exp_voltage_sem = np.asarray(es1_overlay["discharge_voltage_sem_v"], dtype=float)

    current_ax.plot(
        exp_time_ms[visible],
        exp_current[visible],
        color="tab:blue",
        ls="--",
        lw=1.4,
        label="ES1 discharge current",
    )
    current_ax.fill_between(
        exp_time_ms[visible],
        (exp_current - exp_current_sem)[visible],
        (exp_current + exp_current_sem)[visible],
        color="tab:blue",
        alpha=0.14,
        linewidth=0,
    )
    voltage_ax.plot(
        exp_time_ms[visible],
        exp_voltage[visible],
        color="tab:orange",
        ls="--",
        lw=1.4,
        label="ES1 cathode-anode voltage (sign inverted)",
    )
    voltage_ax.fill_between(
        exp_time_ms[visible],
        (exp_voltage - exp_voltage_sem)[visible],
        (exp_voltage + exp_voltage_sem)[visible],
        color="tab:orange",
        alpha=0.14,
        linewidth=0,
    )
    current_handles, current_labels = current_ax.get_legend_handles_labels()
    voltage_handles, voltage_labels = voltage_ax.get_legend_handles_labels()
    current_ax.legend(
        current_handles + voltage_handles,
        current_labels + voltage_labels,
        loc="best",
        fontsize=7,
    )

for name, fig in figures.items():
    if fig is None:
        continue
    # Add port markers only to panels whose x-axis is z [cm] (skips time-axis and colorbar axes).
    for ax in fig.axes:
        if ax.get_xlabel().startswith("z"):
            add_port_lines(ax)
    display(fig)
    plt.close(fig)


## Time-slice profiles at 15 ms and 19 ms

Main-discharge-relative times. Each figure snaps to the nearest saved timestep; the title reports the actual time used. Make sure the run reaches ~19 ms of main discharge, or edit `SLICE_TIMES_MS`.

In [ ]:
SLICE_TIMES_MS = (15.0, 19.0)

for slice_ms in SLICE_TIMES_MS:
    fig = psr._plot_time_slice_summary(
        result=result,
        z_cm=z_cm,
        t_ms=t_slice_ms,
        slice_time_ms=slice_ms,
    )
    for ax in fig.axes:  # all four panels share a z [cm] x-axis
        add_port_lines(ax)
    display(fig)
    plt.close(fig)

## Port time series

`ne(t)` and `Te(t)` at the five port positions (nearest cell to each entry in `ALL_PORT_Z_CM`: dashed ports 21/29/41 and dotted ports 11/50). Solid curves are the simulation and open circles are ES1 core-averaged measurements. The base density uncertainty is radial SEM with Probe A area-calibration uncertainty added in quadrature. Because the experimental inversion assumed cold ions, a simulation-derived finite-$T_i$ systematic is added in quadrature to the **lower** density error only; the upper error remains the base SEM. The correction uses the experimental $T_e$, interpolated simulation $T_i$, and the same `ISAT_GAMMA_I` used below. Temperature error bars are radial SEM. Vertical dashed lines mark the main-discharge/afterglow phase transitions.


In [ ]:
# ne(t) and Te(t) at the port positions, with ES1 core-average overlays.
ne = np.asarray(result.n, dtype=float)
Te = np.asarray(result.Te, dtype=float)
Ti = np.asarray(result.Ti, dtype=float)

exp_ports = np.asarray(es1_overlay["port"], dtype=int)
exp_density_time_ms = (
    np.asarray(es1_overlay["density_time_ms"], dtype=float)
    + ES1_TIME_SHIFT_MS
)
exp_te_time_ms = (
    np.asarray(es1_overlay["te_time_ms"], dtype=float)
    + ES1_TIME_SHIFT_MS
)

fig, axes = plt.subplots(2, 1, figsize=(9, 6), constrained_layout=True)
for port in sorted(ALL_PORT_Z_CM):
    z = ALL_PORT_Z_CM[port]
    idx = int(np.argmin(np.abs(z_cm - z)))  # nearest simulation cell to the port
    exp_idx = int(np.flatnonzero(exp_ports == port)[0])

    density_line, = axes[0].plot(
        t_plot,
        ne[:, idx],
        label=f"sim port {port} (z={z_cm[idx]:.0f} cm)",
    )
    color = density_line.get_color()
    axes[1].plot(
        t_plot,
        Te[:, idx],
        color=color,
        label=f"sim port {port} (z={z_cm[idx]:.0f} cm)",
    )

    exp_te = np.asarray(es1_overlay["te_mean_ev"][exp_idx], dtype=float)
    exp_te_sem = np.asarray(es1_overlay["te_sem_ev"][exp_idx], dtype=float)
    exp_density = np.asarray(es1_overlay["density_mean_cm3"][exp_idx], dtype=float)
    exp_density_base_sem = np.asarray(
        es1_overlay["density_total_sem_cm3"][exp_idx],
        dtype=float,
    )

    # The cold-ion inversion gives n_cold = I/(A*sqrt(Te)). With finite Ti,
    # n = n_cold*sqrt(Te/(Te + gamma_i*Ti)), so the omitted correction is a
    # downward-only systematic. Use measured Te and simulation Ti at this port.
    exp_te_at_density = np.interp(
        exp_density_time_ms,
        exp_te_time_ms,
        exp_te,
        left=np.nan,
        right=np.nan,
    )
    sim_ti_at_density = np.interp(
        exp_density_time_ms,
        t_plot,
        Ti[:, idx],
        left=np.nan,
        right=np.nan,
    )
    finite_ti_sum = exp_te_at_density + ISAT_GAMMA_I * sim_ti_at_density
    valid_ti = (exp_te_at_density > 0.0) & (finite_ti_sum > 0.0)
    density_ti_fraction = np.full_like(exp_density, np.nan)
    density_ti_fraction[valid_ti] = 1.0 - np.sqrt(
        exp_te_at_density[valid_ti] / finite_ti_sum[valid_ti]
    )
    exp_density_ti_systematic = np.abs(exp_density) * density_ti_fraction
    exp_density_lower_error = np.hypot(
        exp_density_base_sem,
        exp_density_ti_systematic,
    )
    exp_density_upper_error = exp_density_base_sem

    density_visible = (
        np.isfinite(exp_density)
        & np.isfinite(exp_density_lower_error)
        & np.isfinite(exp_density_upper_error)
        & (exp_density_time_ms >= np.nanmin(t_plot))
        & (exp_density_time_ms <= np.nanmax(t_plot))
    )
    axes[0].errorbar(
        exp_density_time_ms[density_visible],
        exp_density[density_visible],
        yerr=np.vstack(
            (
                exp_density_lower_error[density_visible],
                exp_density_upper_error[density_visible],
            )
        ),
        fmt="o",
        ls="none",
        ms=3.2,
        mfc="white",
        mec=color,
        mew=0.9,
        ecolor=color,
        elinewidth=0.8,
        capsize=1.5,
        alpha=0.9,
        label=f"ES1 port {port}",
    )

    te_visible = (
        np.isfinite(exp_te)
        & np.isfinite(exp_te_sem)
        & (exp_te_time_ms >= np.nanmin(t_plot))
        & (exp_te_time_ms <= np.nanmax(t_plot))
    )
    axes[1].errorbar(
        exp_te_time_ms[te_visible],
        exp_te[te_visible],
        yerr=exp_te_sem[te_visible],
        fmt="o",
        ls="none",
        ms=3.2,
        mfc="white",
        mec=color,
        mew=0.9,
        ecolor=color,
        elinewidth=0.8,
        capsize=1.5,
        alpha=0.9,
        label=f"ES1 port {port}",
    )

axes[0].set_ylabel(r"$n_e$ [cm$^{-3}$]")
axes[0].set_yscale("linear")
axes[0].set_title(
    rf"Electron density at ports (lower error includes $T_i$, $\gamma_i={ISAT_GAMMA_I:g}$)"
)
axes[1].set_ylabel(r"$T_e$ [eV]")
axes[1].set_title("Electron temperature at ports")
axes[0].set_ylim(0, 2e13)
axes[1].set_ylim(0, 12)
for ax in axes:
    ax.set_xlabel(time_label)
    ax.grid(True, alpha=0.3)
    ax.legend(loc="best", fontsize=7, ncol=2)
    psr._add_phase_lines(ax, phase_events)  # main_discharge / afterglow markers

fig.suptitle(f"Port time series\n{psr._plot_title(result)}", fontsize=11)
display(fig)
plt.close(fig)


## Normalized upstream Isat decay in the afterglow

Each panel compares the offset-corrected, positive upstream-face ES1 ion-saturation current at `x=0` with simulation proxies at the corresponding port. Experimental shots flagged as high-current outliers in any pre-afterglow inter-sweep cycle at `x=0` are excluded. The experimental curve and its shot SEM are divided by the interpolated experimental mean at afterglow start.

The requested cold-ion simulation proxy, $n_e\sqrt{T_e}$, is shown together with the finite-ion-temperature proxy $n_e\sqrt{T_e + \gamma_i T_i}$. Both are independently normalized at the exact simulation afterglow transition. Set `ISAT_GAMMA_I` above to test a different ion closure; `0` reproduces the cold-ion assumption.


In [ ]:
# Normalized upstream Isat decay: ES1 measurements vs sim1d proxies.
Ti = np.asarray(result.Ti, dtype=float)
afterglow_events_ms = [
    float(time_ms)
    for time_ms, phase in phase_events
    if phase == "afterglow"
]
if not afterglow_events_ms:
    raise RuntimeError("Simulation result has no afterglow phase transition")
afterglow_start_ms = afterglow_events_ms[0]
simulation_end_ms = float(np.nanmax(t_plot))
simulation_afterglow = (t_plot >= afterglow_start_ms) & (t_plot <= simulation_end_ms)
simulation_elapsed_ms = t_plot[simulation_afterglow] - afterglow_start_ms

exp_isat_time_ms = (
    np.asarray(es1_overlay["isat_decay_time_ms"], dtype=float)
    + ES1_TIME_SHIFT_MS
)
exp_isat_ports = np.asarray(es1_overlay["isat_decay_port"], dtype=int)

fig, axes = plt.subplots(
    len(ALL_PORT_Z_CM),
    1,
    figsize=(9, 11),
    sharex=True,
    constrained_layout=True,
)
for ax, port in zip(axes, sorted(ALL_PORT_Z_CM)):
    z = ALL_PORT_Z_CM[port]
    sim_idx = int(np.argmin(np.abs(z_cm - z)))
    exp_idx = int(np.flatnonzero(exp_isat_ports == port)[0])

    sim_cold = ne[:, sim_idx] * np.sqrt(np.maximum(Te[:, sim_idx], 0.0))
    sim_finite_ti = ne[:, sim_idx] * np.sqrt(
        np.maximum(Te[:, sim_idx] + ISAT_GAMMA_I * Ti[:, sim_idx], 0.0)
    )
    cold_norm = float(np.interp(afterglow_start_ms, t_plot, sim_cold))
    finite_ti_norm = float(np.interp(afterglow_start_ms, t_plot, sim_finite_ti))
    if cold_norm <= 0.0 or finite_ti_norm <= 0.0:
        raise ValueError(f"Non-positive simulation Isat normalization at port {port}")

    exp_mean = np.asarray(es1_overlay["isat_decay_mean_a"][exp_idx], dtype=float)
    exp_sem = np.asarray(es1_overlay["isat_decay_sem_a"][exp_idx], dtype=float)
    exp_norm = float(np.interp(afterglow_start_ms, exp_isat_time_ms, exp_mean))
    if exp_norm <= 0.0:
        raise ValueError(f"Non-positive experimental Isat normalization at port {port}")
    exp_elapsed_ms = exp_isat_time_ms - afterglow_start_ms
    exp_visible = (
        np.isfinite(exp_mean)
        & np.isfinite(exp_sem)
        & (exp_elapsed_ms >= 0.0)
        & (exp_isat_time_ms <= simulation_end_ms)
    )
    visible_idx = np.flatnonzero(exp_visible)
    error_idx = visible_idx[::ISAT_ERRORBAR_STRIDE]

    ax.plot(
        simulation_elapsed_ms,
        sim_cold[simulation_afterglow] / cold_norm,
        color="0.45",
        ls="--",
        lw=1.4,
        label=r"simulation $n_e\sqrt{T_e}$",
    )
    ax.plot(
        simulation_elapsed_ms,
        sim_finite_ti[simulation_afterglow] / finite_ti_norm,
        color="tab:blue",
        lw=1.7,
        label=rf"simulation $n_e\sqrt{{T_e + {ISAT_GAMMA_I:g}T_i}}$",
    )
    ax.plot(
        exp_elapsed_ms[visible_idx],
        exp_mean[visible_idx] / exp_norm,
        color="tab:red",
        lw=1.1,
        alpha=0.8,
        label="ES1 upstream Isat",
    )
    ax.errorbar(
        exp_elapsed_ms[error_idx],
        exp_mean[error_idx] / exp_norm,
        yerr=exp_sem[error_idx] / exp_norm,
        fmt="o",
        ls="none",
        ms=2.8,
        mfc="white",
        mec="tab:red",
        ecolor="tab:red",
        elinewidth=0.75,
        capsize=1.2,
        alpha=0.85,
        label="ES1 shot SEM",
    )
    ax.axhline(1.0, color="0.65", lw=0.7, ls=":")
    ax.set_ylabel("Normalized\nIsat proxy")
    ax.set_title(
        f"Port {port} (simulation z={z_cm[sim_idx]:.0f} cm; "
        f"ES1 n={int(es1_overlay['isat_decay_n_shots_used'][exp_idx])} shots)",
        fontsize=9,
    )
    ax.set_ylim(bottom=0.0)
    ax.grid(True, alpha=0.25)

axes[0].legend(loc="best", fontsize=7, ncol=2)
axes[-1].set_xlabel("Time since afterglow start [ms]")
axes[-1].set_xlim(0.0, simulation_end_ms - afterglow_start_ms)
fig.suptitle(
    "ES1 x=0 upstream ion-current decay vs sim1d\n"
    + psr._plot_title(result),
    fontsize=11,
)
display(fig)
plt.close(fig)


## Heat-term time slices

A curated set of the most important energy source terms, each as a line of its **absolute value** in W/cm² vs z (converted from the stored W/cm³ via `length_cm`), with **▲ markers where the term heats** and **▼ where it cools** that species. Terms are grouped in `HEAT_TERM_GROUPS` — e.g. **net beam** = deposition + birth + cost, **recombination** = radiative + 3-body — and surface/end-loss and pure transport terms are dropped. Same slice times as above (`SLICE_TIMES_MS`); groups that are ~0 for a species are omitted. Edit `HEAT_TERM_GROUPS` to add/remove terms. Styled after `bapsf_app`'s heat-term slices.

In [ ]:
# Heat-term time slices (bapsf-app style): |grouped source term| in W/cm^2 vs z,
# with up markers where the term heats and down markers where it cools.
length_cm = np.asarray(result.length_cm, dtype=float)  # dz = V_cell / A_plasma
HEAT_FLOOR_W_CM2 = 1e-9  # ignore terms below this; also the log-axis floor

# Curated groups of the most important terms; each label sums its signed
# component term keys (stored W/cm^3). Components that are ~0 for a species drop
# out automatically, so the same set works for both panels. Surface/end-loss and
# pure transport (advective/front flux, pressure work) terms are omitted here --
# see the volume-integrated budget below for those.
HEAT_TERM_GROUPS = {
    "net beam": ["beam_power_deposition", "beam_ionization_birth", "beam_ionization_cost"],
    "e-i exchange": ["ei_exchange"],
    "electron-ion cooling": ["electron_ion_cooling"],
    "electron-neutral cooling": ["electron_neutral_cooling"],
    "ion charge exchange (Qcx)": ["ion_charge_exchange"],
    "ion-neutral thermalization": ["ion_neutral_thermalization"],
    "heat conduction": ["heat_conduction"],
    "recombination": ["recombination_rad_loss", "recombination_3b_loss"],
}


def plot_signed_abs(ax, z, y, label, color):
    """abs(y) line; up-triangle where y>0 (heating), down-triangle where y<0 (cooling)."""
    yabs = np.abs(y)
    ax.plot(z, yabs, color=color, lw=1.2, alpha=0.7, zorder=2)
    pos = y >= 0
    ax.scatter(z[pos], yabs[pos], color=color, marker="+", s=24, zorder=3, label=label)
    ax.scatter(z[~pos], yabs[~pos], color=color, marker="_", s=24, zorder=3)


def plot_heat_terms(ax, terms_dict, idx, title):
    colors = plt.rcParams["axes.prop_cycle"].by_key()["color"]
    ci = 0
    for label, components in HEAT_TERM_GROUPS.items():
        q = np.zeros_like(length_cm)  # net signed W/cm^2 for the group
        for name in components:
            if name in terms_dict:
                q = q + np.asarray(terms_dict[name], dtype=float)[idx, :] * length_cm
        if not np.any(np.abs(q) > HEAT_FLOOR_W_CM2):
            continue  # group inactive for this species
        plot_signed_abs(ax, z_cm, q, label, colors[ci % len(colors)])
        ci += 1
    add_port_lines(ax)
    ax.set_yscale("log")
    ax.set_ylim(bottom=HEAT_FLOOR_W_CM2)
    ax.set_xlabel("z [cm]")
    ax.set_ylabel(r"$|q|$ [W cm$^{-2}$]")
    ax.set_title(title)
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=7, loc="best", ncol=2)


for slice_ms in SLICE_TIMES_MS:
    idx = int(np.argmin(np.abs(t_slice_ms - slice_ms)))
    actual_ms = float(t_slice_ms[idx])
    fig, axes = plt.subplots(1, 2, figsize=(13, 5), constrained_layout=True)
    plot_heat_terms(axes[0], result.electron_energy_terms_W_cm3, idx, "Electron heat terms")
    plot_heat_terms(axes[1], result.ion_energy_terms_W_cm3, idx, "Ion heat terms")
    fig.suptitle(
        f"Heat-term slice at {slice_ms:.1f} ms (nearest saved {actual_ms:.3f} ms)"
        "   ▲ heating / ▼ cooling\n" + psr._plot_title(result),
        fontsize=11,
    )
    display(fig)
    plt.close(fig)

## Volume-integrated energy budget

Every energy source term integrated over the plasma volume, in **watts vs time** — the same terms as the slice plots above, but summed over the whole column so the totals are directly comparable and the discharge's power balance is visible in one place. Unlike the slice plots, this **includes surface/end loss** and the transport terms.

Signs are physical: **positive heats** that species, **negative cools** it. The y-axis is symlog, so terms spanning many orders are all visible while the sign is preserved; the shaded band is the linear region near zero. Terms whose peak never exceeds `BUDGET_LINTHRESH_W` are dropped from the legend as inactive for that species.

**The net gets its own panel.** It is a near-cancellation of terms that individually reach ~1e5 W, so on a shared axis it is unreadable — the bottom panel gives it its own scale, with the electron net, the ion net, and their total.

That panel also carries a **completeness check**: the dotted green line is the finite-difference `d(thermal energy)/dt` from the saved trajectory, computed independently of the terms. It should track the total net. Where it doesn't, the gap is energy the *floors* injected — which no term reports — plus time-discretization error between saved samples. A persistent divergence would mean a source term is missing from `BUDGET_GROUPS`.

**Conservation check.** `heat_conduction`, `plasma_advective_flux` and `plasma_front_flux` are flux *divergences* over a closed domain: they only move energy between cells, so their volume integral is analytically zero. They are printed below rather than plotted — a line at ~1e-9 W tells you nothing on a plot whose other terms are ~1e5 W — and how close they land to zero, relative to the beam power, checks the conservative flux implementation. They are still counted in the net. Pressure work is *not* one of these: it is a genuine source and stays in the plot.

In [ ]:
# Volume-integrated energy budget [W] vs time.
# Terms are stored as W/cm^3 per cell; multiplying by the plasma cell volume and
# summing over z gives the total power each term puts into (or takes out of) the
# column. Sign is physical: positive heats that species.
Vp = np.asarray(result.plasma_volume_cm3, dtype=float)

BUDGET_LINTHRESH_W = 1.0  # symlog linear region [W]; also the "inactive" cutoff
NET_LINTHRESH_W = 1.0     # symlog linear region for the net panel [W]

# Grouped like HEAT_TERM_GROUPS, but adding what the slice plots omit: the
# surface/end loss and the transport terms.
BUDGET_GROUPS = {
    "beam power (incl. ohmic)": ["beam_power_deposition"],
    "beam ionization": ["beam_ionization_birth", "beam_ionization_cost"],
    "bulk ionization": ["ionization_birth", "ionization_energy_cost"],
    "e-i exchange": ["ei_exchange"],
    "electron-ion cooling": ["electron_ion_cooling"],
    "electron-neutral cooling": ["electron_neutral_cooling"],
    "ion charge exchange (Qcx)": ["ion_charge_exchange"],
    "ion-neutral thermalization": ["ion_neutral_thermalization"],
    "ion-neutral friction": ["ion_neutral_frictional_heating"],
    "recombination": ["recombination_rad_loss", "recombination_3b_loss"],
    "surface/end loss": ["surface_loss", "cathode_surface_loss"],
    "pressure work": ["pressure_work"],
}

# Flux divergences over a closed domain: these only move energy between cells, so
# their volume integral must vanish. Reported as a conservation check below
# rather than plotted, since a line at ~1e-9 W is not informative on a plot whose
# other terms are ~1e5 W. Pressure work is NOT one of these -- it is a genuine
# source and stays in BUDGET_GROUPS above. They are still counted in the net.
CONSERVATION_CHECKS = {
    "heat conduction": ["heat_conduction"],
    "advective flux": ["plasma_advective_flux"],
    "front flux": ["plasma_front_flux"],
}


def integrate_W(terms_dict, names):
    """Volume-integrate the named W/cm^3 terms over the plasma -> W vs time."""
    total = np.zeros(len(result.time), dtype=float)
    for name in names:
        if name in terms_dict:
            total = total + np.sum(
                np.asarray(terms_dict[name], dtype=float) * Vp[None, :], axis=1
            )
    return total


def plot_budget(ax, terms_dict, title):
    """Draw the individual terms; return the net for the separate net panel."""
    colors = plt.rcParams["axes.prop_cycle"].by_key()["color"]
    net = np.zeros(len(result.time), dtype=float)
    ci = 0
    for label, components in {**BUDGET_GROUPS, **CONSERVATION_CHECKS}.items():
        p_W = integrate_W(terms_dict, components)
        net = net + p_W  # the net counts the flux terms too, even though they are ~0
        if label in CONSERVATION_CHECKS or np.max(np.abs(p_W)) < BUDGET_LINTHRESH_W:
            continue  # inactive for this species; keeps the legend readable
        ax.plot(t_plot, p_W, color=colors[ci % len(colors)], lw=1.3, label=label)
        ci += 1
    ax.axhline(0.0, color="gray", lw=0.8, alpha=0.6)
    ax.axhspan(-BUDGET_LINTHRESH_W, BUDGET_LINTHRESH_W, color="gray", alpha=0.12, lw=0)
    ax.set_yscale("symlog", linthresh=BUDGET_LINTHRESH_W)
    ax.set_ylabel("power [W]")
    ax.set_title(title)
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=7, loc="best", ncol=2)
    psr._add_phase_lines(ax, phase_events)
    return net


def plot_net(ax, net_e, net_i):
    """Net power per species, on its own axis.

    The net is a near-cancellation of terms that individually reach ~1e5 W, so
    sharing an axis with them makes it unreadable. Here it gets its own scale.
    """
    ax.plot(t_plot, net_e, color="tab:blue", lw=1.4, label="electron net")
    ax.plot(t_plot, net_i, color="tab:red", lw=1.4, label="ion net")
    ax.plot(t_plot, net_e + net_i, color="k", lw=1.8, ls="--", label="total net")

    # Independent check on whether the term list is complete: the finite-difference
    # rate of change of the actual thermal energy should track the total net.
    # Where it does not, the gap is energy the floors injected -- which no term
    # reports -- plus the time-discretization error between saved samples.
    E_tot_erg = np.sum(
        (np.asarray(result.Ee, dtype=float) + np.asarray(result.Ei, dtype=float))
        * Vp[None, :],
        axis=1,
    )
    dEdt_W = np.gradient(E_tot_erg, np.asarray(result.time, dtype=float)) * 1.0e-7
    ax.plot(t_plot, dEdt_W, color="tab:green", lw=1.0, ls=":", alpha=0.9,
            label="d(thermal)/dt  [finite diff]")

    ax.axhline(0.0, color="gray", lw=0.8, alpha=0.6)
    ax.axhspan(-NET_LINTHRESH_W, NET_LINTHRESH_W, color="gray", alpha=0.12, lw=0)
    ax.set_yscale("symlog", linthresh=NET_LINTHRESH_W)
    ax.set_xlabel(time_label)
    ax.set_ylabel("power [W]")
    ax.set_title("Net power  (+ heats / - cools)")
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=7, loc="best", ncol=2)
    psr._add_phase_lines(ax, phase_events)


fig, axes = plt.subplots(3, 1, figsize=(11, 13), constrained_layout=True, sharex=True)
net_e = plot_budget(axes[0], result.electron_energy_terms_W_cm3,
                    "Electron energy budget (volume-integrated)")
net_i = plot_budget(axes[1], result.ion_energy_terms_W_cm3,
                    "Ion energy budget (volume-integrated)")
plot_net(axes[2], net_e, net_i)
fig.suptitle(
    "Volume-integrated energy budget   (+ heats / - cools)\n" + psr._plot_title(result),
    fontsize=11,
)
display(fig)
plt.close(fig)

# Peak magnitude of each term: a quick numeric read of who dominates.
print(f"{'term':32} {'|peak| electron [W]':>20} {'|peak| ion [W]':>18}")
for label, components in BUDGET_GROUPS.items():
    pe = np.max(np.abs(integrate_W(result.electron_energy_terms_W_cm3, components)))
    pi = np.max(np.abs(integrate_W(result.ion_energy_terms_W_cm3, components)))
    if max(pe, pi) < BUDGET_LINTHRESH_W:
        continue
    print(f"  {label:30} {pe:20.4g} {pi:18.4g}")
print(f"  {'NET':30} {np.max(np.abs(net_e)):20.4g} {np.max(np.abs(net_i)):18.4g}")

# Conservation check: each of these is a flux divergence over a closed domain,
# so its volume integral is zero analytically. How close it lands to zero,
# relative to the beam power driving the discharge, is a check on the
# conservative flux implementation -- not a physics result.
beam_scale = max(
    np.max(np.abs(integrate_W(result.electron_energy_terms_W_cm3, ["beam_power_deposition"]))),
    1.0,
)
print(f"\nconservation checks (volume integral of a flux divergence must vanish;"
      f" beam power = {beam_scale:.3g} W):")
for label, components in CONSERVATION_CHECKS.items():
    pe = np.max(np.abs(integrate_W(result.electron_energy_terms_W_cm3, components)))
    pi = np.max(np.abs(integrate_W(result.ion_energy_terms_W_cm3, components)))
    print(f"  {label:18} |peak| e {pe:10.3e} W   i {pi:10.3e} W"
          f"   -> {max(pe, pi) / beam_scale:.1e} of beam power")

## True heating vs true loss

A focused view of the *physical* power balance: only the terms that genuinely put energy **into** a species (heating) or take it **out** (loss), plotted as **absolute value in watts** so the magnitudes sit against each other on one log axis. This drops the transport/flux divergences and the birth/cost bookkeeping that the full budget above carries, and it **splits the cathode-boundary electron loss into its cathode-sheath and anode-sheath parts** — the volumetric `cathode_surface_loss` term bundles them, so the split comes from the scalar circuit diagnostics (`P_cathode_e`, `P_anode_e`), whose sum reproduces that term.

- **Electrons** — heating: beam + ohmic deposited at/near the cathode. Losses: **Qie** (transfer to ions), **Qei** (electron–ion inelastic/radiative cooling), **Qen** (electron–neutral cooling), **end/surface loss**, **cathode-sheath loss**, **anode-sheath loss**.
- **Ions** — heating: **Qie** received from the electrons. Losses: **Qcx** (charge exchange), **ion–neutral thermalization**, **surface/end loss**.

Two electron-only companion panels accompany the magnitudes:

- **Losses as % of injected source power** (`P_prim + P_ohmic` from the cathode solve), blanked where the cathode delivers negligible power so the ratio doesn't blow up.
- **Injected vs deposited power** — injected is what the cathode circuit delivers; deposited is the volume integral of the beam/ohmic term. The shaded gap is beam power that crosses the column without being absorbed (the beam-bypass fraction, which is large in this run).

Sign conventions follow the terms: Qie is a loss on electrons and the matching gain on ions when `Te > Ti`; if the sign ever flips (`Ti > Te`) the magnitude is still what's drawn.

In [ ]:
# --- True heating vs true loss: electrons -----------------------------------
# Only the genuine heating/loss channels, as |power| [W] so magnitudes are
# comparable on one log axis. Reuses integrate_W / Vp / t_plot from the budget
# cell above.
FLOOR_W = 1.0  # log-axis floor / inactivity cutoff [W]
et = result.electron_energy_terms_W_cm3

# Cathode circuit scalar diagnostics [W] (already SI watts). NaN marks phases
# where the cathode is not being solved (pre-breakdown / afterglow) -> no power
# that phase. Source + (twin) end cathodes are summed.
cd = getattr(result, "cathode_diagnostics", {}) or {}


def cathode_power_W(name):
    """Sum the source and (twin) end cathode scalar diagnostic [W], NaN -> 0."""
    total = np.zeros(len(result.time), dtype=float)
    for prefix in ("source", "end"):
        key = f"{prefix}_{name}"
        if key in cd:
            total = total + np.nan_to_num(np.asarray(cd[key], dtype=float), nan=0.0)
    return total


P_prim = cathode_power_W("P_prim")        # primary-beam power into the column
P_ohmic = cathode_power_W("P_ohmic")      # ohmic dissipation at the cathode cell
P_cathode_e = cathode_power_W("P_cathode_e")  # electron power to the cathode sheath
P_anode_e = cathode_power_W("P_anode_e")      # electron power to the anode sheath
P_injected = P_prim + P_ohmic

# Deposited electron heating: what the beam/ohmic term actually lands in the
# column (absorbed primary beam along the Beer-Lambert profile + ohmic at the
# cathode cell). The volumetric term integrates to <= P_injected; the shortfall
# is beam that crosses the column without being absorbed.
P_deposited = integrate_W(et, ["beam_power_deposition"])

# Loss channels as signed W (negative = a loss on electrons); the cathode/anode
# scalars are already positive losses. Plotted as |.|. Cathode- and anode-sheath
# losses come from the scalar diagnostics because the volumetric
# `cathode_surface_loss` term bundles the two together
# (-cathode_surface_loss.Ee integrates to P_cathode_e + P_anode_e).
elec_losses = {
    "Qie (to ions)":           integrate_W(et, ["ei_exchange"]),
    "Qei (e-ion cooling)":     integrate_W(et, ["electron_ion_cooling"]),
    "Qen (e-neutral cooling)": integrate_W(et, ["electron_neutral_cooling"]),
    "end/surface loss":        integrate_W(et, ["surface_loss"]),
    "cathode-sheath loss":     -P_cathode_e,
    "anode-sheath loss":       -P_anode_e,
}
elec_loss_total = sum(np.abs(y) for y in elec_losses.values())

fig, axes = plt.subplots(3, 1, figsize=(11, 13), constrained_layout=True, sharex=True)
colors = plt.rcParams["axes.prop_cycle"].by_key()["color"]

# Panel 1: |power| magnitudes, heating vs losses on one log axis.
ax = axes[0]
ax.plot(t_plot, np.abs(P_deposited), color="k", lw=2.2, label="cathode heating (deposited)")
for i, (label, y) in enumerate(elec_losses.items()):
    ax.plot(t_plot, np.abs(y), color=colors[i % len(colors)], lw=1.3, label=label)
ax.plot(t_plot, elec_loss_total, color="0.4", lw=1.6, ls="--", label="total losses")
ax.set_yscale("log")
ax.set_ylim(bottom=FLOOR_W)
ax.set_ylabel("|power| [W]")
ax.set_title("Electron heating vs loss  (|magnitude|)")
ax.grid(True, alpha=0.3)
ax.legend(fontsize=7, loc="best", ncol=2)
psr._add_phase_lines(ax, phase_events)

# Panel 2: losses as % of injected source power (ohmic + beam). Blanked where the
# cathode delivers negligible power, so the ratio does not blow up.
ax = axes[1]
inj_floor = 0.01 * np.max(P_injected) if np.max(P_injected) > 0 else np.inf
denom = np.where(P_injected > inj_floor, P_injected, np.nan)
for i, (label, y) in enumerate(elec_losses.items()):
    ax.plot(t_plot, 100.0 * np.abs(y) / denom, color=colors[i % len(colors)], lw=1.3,
            label=label)
ax.plot(t_plot, 100.0 * elec_loss_total / denom, color="0.4", lw=1.6, ls="--",
        label="total losses")
ax.axhline(100.0, color="gray", lw=0.8, alpha=0.6)
ax.set_ylabel("loss / injected  [%]")
ax.set_title("Electron losses as % of injected source power (ohmic + beam)")
ax.grid(True, alpha=0.3)
ax.legend(fontsize=7, loc="best", ncol=2)
psr._add_phase_lines(ax, phase_events)

# Panel 3: injected vs deposited power. The gap is beam power that crosses the
# column without being absorbed (the beam-bypass fraction).
ax = axes[2]
ax.plot(t_plot, P_injected, color="tab:blue", lw=1.6, label="injected (P_prim + P_ohmic)")
ax.plot(t_plot, P_deposited, color="tab:green", lw=1.6, label="deposited (beam+ohmic term)")
ax.plot(t_plot, P_ohmic, color="tab:orange", lw=1.1, ls=":", label="ohmic only")
ax.fill_between(t_plot, P_deposited, P_injected, where=(P_injected >= P_deposited),
                color="tab:red", alpha=0.15, label="un-deposited (beam bypass)")
ax.set_xlabel(time_label)
ax.set_ylabel("power [W]")
ax.set_title("Injected vs deposited power")
ax.grid(True, alpha=0.3)
ax.legend(fontsize=7, loc="best")
psr._add_phase_lines(ax, phase_events)

fig.suptitle("Electron true heating vs loss\n" + psr._plot_title(result), fontsize=11)
display(fig)
plt.close(fig)

# Numeric read: peak of each channel. The "% of inj peak" column divides each
# channel's own peak by the peak injected power, so it is a rough magnitude
# ranking, not a same-instant ratio (peaks need not coincide).
inj_peak = np.max(P_injected)
inj_peak = inj_peak if inj_peak > 0 else np.nan
print(f"peak injected source power (ohmic+beam): {inj_peak:.4g} W")
print(f"{'electron channel':30} {'|peak| [W]':>14} {'% of inj peak':>14}")
print(f"  {'cathode heating (deposited)':28} {np.max(np.abs(P_deposited)):14.4g} "
      f"{100 * np.max(np.abs(P_deposited)) / inj_peak:13.1f}")
for label, y in elec_losses.items():
    print(f"  {label:28} {np.max(np.abs(y)):14.4g} {100 * np.max(np.abs(y)) / inj_peak:13.1f}")
print(f"  {'total losses':28} {np.max(elec_loss_total):14.4g} "
      f"{100 * np.max(elec_loss_total) / inj_peak:13.1f}")

In [ ]:
# --- True heating vs true loss: ions ----------------------------------------
# Heating is Qie received from the electrons; losses are charge exchange,
# ion-neutral thermalization, and the surface/end sinks. Panel 1 is |power| [W]
# on one log axis; panel 2 is the losses as % of the ion heating (Qie), the
# mirror of the electron % panel. Reuses integrate_W / t_plot from above.
FLOOR_W = 1.0  # log-axis floor [W]
it = result.ion_energy_terms_W_cm3

Qie_heating = integrate_W(it, ["ei_exchange"])  # ion heating from the electrons
ion_losses = {
    "Qcx (charge exchange)":      -integrate_W(it, ["ion_charge_exchange"]),
    "ion-neutral thermalization": -integrate_W(it, ["ion_neutral_thermalization"]),
    # cathode_surface_loss carries the ion energy dumped at the cathode surface;
    # surface_loss carries the source/end-wall neutralization. Grouped as one
    # surface/end sink here.
    "surface/end loss":           -integrate_W(it, ["surface_loss", "cathode_surface_loss"]),
}
ion_loss_total = sum(np.abs(y) for y in ion_losses.values())

fig, axes = plt.subplots(2, 1, figsize=(11, 9), constrained_layout=True, sharex=True)
colors = plt.rcParams["axes.prop_cycle"].by_key()["color"]

# Panel 1: |power| magnitudes, heating vs losses.
ax = axes[0]
ax.plot(t_plot, np.abs(Qie_heating), color="k", lw=2.2,
        label="Qie (from electrons)  [heating]")
for i, (label, y) in enumerate(ion_losses.items()):
    ax.plot(t_plot, np.abs(y), color=colors[i % len(colors)], lw=1.3, label=label)
ax.plot(t_plot, ion_loss_total, color="0.4", lw=1.6, ls="--", label="total losses")
ax.set_yscale("log")
ax.set_ylim(bottom=FLOOR_W)
ax.set_ylabel("|power| [W]")
ax.set_title("Ion heating vs loss  (|magnitude|)")
ax.grid(True, alpha=0.3)
ax.legend(fontsize=8, loc="best", ncol=2)
psr._add_phase_lines(ax, phase_events)

# Panel 2: losses as % of the ion heating (Qie). Blanked where Qie delivers
# negligible power, so the ratio does not blow up. Unlike the electron panel the
# denominator is the *only* heating channel, so the total can and does exceed
# 100% wherever the ion thermal energy is falling.
ax = axes[1]
heat_mag = np.abs(Qie_heating)
heat_floor = 0.01 * np.max(heat_mag) if np.max(heat_mag) > 0 else np.inf
denom = np.where(heat_mag > heat_floor, heat_mag, np.nan)
for i, (label, y) in enumerate(ion_losses.items()):
    ax.plot(t_plot, 100.0 * np.abs(y) / denom, color=colors[i % len(colors)], lw=1.3,
            label=label)
ax.plot(t_plot, 100.0 * ion_loss_total / denom, color="0.4", lw=1.6, ls="--",
        label="total losses")
ax.axhline(100.0, color="gray", lw=0.8, alpha=0.6)
ax.set_xlabel(time_label)
ax.set_ylabel("loss / Qie heating  [%]")
ax.set_title("Ion losses as % of ion heating (Qie)")
ax.grid(True, alpha=0.3)
ax.legend(fontsize=8, loc="best", ncol=2)
psr._add_phase_lines(ax, phase_events)

fig.suptitle("Ion true heating vs loss\n" + psr._plot_title(result), fontsize=11)
display(fig)
plt.close(fig)

qie_peak = np.max(heat_mag)
qie_peak = qie_peak if qie_peak > 0 else np.nan
print(f"peak ion heating (Qie): {qie_peak:.4g} W")
print(f"{'ion channel':30} {'|peak| [W]':>14} {'% of Qie peak':>14}")
print(f"  {'Qie (from electrons)':28} {np.max(heat_mag):14.4g} {100.0:13.1f}  [heating]")
for label, y in ion_losses.items():
    print(f"  {label:28} {np.max(np.abs(y)):14.4g} {100 * np.max(np.abs(y)) / qie_peak:13.1f}")
print(f"  {'total losses':28} {np.max(ion_loss_total):14.4g} "
      f"{100 * np.max(ion_loss_total) / qie_peak:13.1f}")

## Power efficiency

The electrical power chain from the cathode bank into the plasma column, and how much survives each stage:

$$P_\text{wall} = V_\text{bank}\,I_\text{tot} \;\xrightarrow{-\,P_\text{comp}}\; P_\text{load} = V_\text{bias}\,I_\text{tot} \;\xrightarrow{-\,\text{sheath}}\; P_\text{inj} = P_\text{prim}+P_\text{ohmic} \;\xrightarrow{-\,\text{bypass}}\; P_\text{dep} \;\xrightarrow{-\,\text{e sheath}}\; P_\text{net}$$

- **P_wall** — power drawn from the cathode bank, $V_\text{bank} I_\text{tot}$.
- **P_load** — power delivered across the anode–cathode gap, $V_\text{bias} I_\text{tot}$, where $V_\text{bias} = V_\text{bank} - I_\text{tot} R_\text{comp}$.
- **P_comp** — dissipated in the compensation resistor, $I_\text{tot}^2 R_\text{comp}$ (this is the resistor dissipation; the drop from wall to load). *Note your message wrote this as $I_\text{tot} R_\text{comp}^2$ — the stored/plotted value is the physically correct $I^2R$.*
- **P_injected** — power injected into the plasma, primary beam + ohmic ($P_\text{prim}+P_\text{ohmic}$).
- **P_deposited** — power actually deposited in the column (volume integral of the beam/ohmic deposition term); below injected by the un-absorbed beam-bypass fraction.
- **P_net** — net power retained by the plasma, $P_\text{inj} - P_\text{cathode,e} - P_\text{anode,e}$. This subtracts **only the electron cathode + anode sheath losses**, the ones this run actually applies (together they equal the volumetric `cathode_surface_loss` term). It is **not** the solver's stored `P_net`/`P_net2`/`P_loss`, which also carry the ion-sheath (`_pl`) terms that aren't deposited as a loss here.

Three panels: **raw power** [W] for all quantities; **wall efficiency** (each downstream stage as % of `P_wall`, with the `R_comp` loss excluded as a line since it is what makes load < wall); and **load efficiency** (as % of `P_load`, with both the `R_comp` loss and the wall stage excluded as upstream of the load). All cathode scalars sum the source and twin end cathode, with NaN (cathode-off phases) treated as zero power.

In [ ]:
# --- Power efficiency: wall -> load -> injected -> deposited -> net -----------
# The electrical power chain from the cathode bank into the plasma column:
#
#   P_wall  = V_bank * I_tot          power drawn from the cathode bank
#     |  - P_comp = I_tot^2 * R_comp  dissipated in the compensation resistor
#   P_load  = V_bias * I_tot          delivered across the anode-cathode gap
#     |  - sheath / structure losses
#   P_inj   = P_prim + P_ohmic        injected into the plasma (beam + ohmic)
#     |  - beam bypass (un-absorbed primaries)
#   P_dep   = beam_power_deposition   actually deposited in the column
#     |  - electron cathode + anode sheath losses
#   P_net   = P_inj - P_cathode_e - P_anode_e   net retained by the plasma
#
# All cathode scalars are stored per timestep (source + twin end cathode);
# P_comp is the stored I_tot^2 * R_comp resistor dissipation. Reuses integrate_W
# / cathode_power_W / t_plot from the cells above.
P_wall = cathode_power_W("P_wall")                       # V_bank * I_tot
P_load = cathode_power_W("P_load")                       # V_bias * I_tot
P_comp = cathode_power_W("P_comp")                       # I_tot^2 * R_comp
P_injected = cathode_power_W("P_prim") + cathode_power_W("P_ohmic")
P_deposited = integrate_W(result.electron_energy_terms_W_cm3, ["beam_power_deposition"])

# Net power retained by the plasma = injected minus the sheath losses this sim
# actually applies. We deliberately do NOT use the solver's stored P_net/P_net2
# (or P_loss): P_loss also carries the ion-sheath terms (P_cathode_i_pl,
# P_anode_i_pl), which are not deposited as a loss in this run's energy budget.
# The electron cathode + anode sheath powers are the only sheath losses applied
# here (together they equal the volumetric cathode_surface_loss term).
P_elec_sheath = cathode_power_W("P_cathode_e") + cathode_power_W("P_anode_e")
P_net = P_injected - P_elec_sheath

fig, axes = plt.subplots(3, 1, figsize=(11, 13), constrained_layout=True, sharex=True)

# Panel 1: raw power [W] for every stage of the chain.
ax = axes[0]
ax.plot(t_plot, P_wall, color="k", lw=2.0, label="P_wall (V_bank * I_tot)")
ax.plot(t_plot, P_load, color="tab:blue", lw=1.6, label="P_load (V_bias * I_tot)")
ax.plot(t_plot, P_injected, color="tab:green", lw=1.6, label="P_injected (ohmic + beam)")
ax.plot(t_plot, P_deposited, color="tab:purple", lw=1.6, label="P_deposited (in column)")
ax.plot(t_plot, P_net, color="tab:brown", lw=1.6, label="P_net (inj - e sheath)")
ax.plot(t_plot, P_comp, color="tab:red", lw=1.3, ls=":", label="P_comp (I_tot^2 * R_comp)")
ax.set_ylabel("power [W]")
ax.set_title("Power chain: wall -> load -> injected -> deposited -> net")
ax.grid(True, alpha=0.3)
ax.legend(fontsize=8, loc="best", ncol=2)
psr._add_phase_lines(ax, phase_events)


def _pct_denom(p, frac=0.01):
    """Blank the denominator where the reference power is negligible."""
    peak = np.max(p)
    floor = frac * peak if peak > 0 else np.inf
    return np.where(p > floor, p, np.nan)


# Panel 2: wall efficiency -- everything downstream as % of P_wall. R_comp is the
# loss that makes P_load < P_wall, so it is excluded as a line (P_wall = 100%).
ax = axes[1]
denom_wall = _pct_denom(P_wall)
ax.plot(t_plot, 100.0 * P_load / denom_wall, color="tab:blue", lw=1.6,
        label="load / wall")
ax.plot(t_plot, 100.0 * P_injected / denom_wall, color="tab:green", lw=1.6,
        label="injected / wall")
ax.plot(t_plot, 100.0 * P_deposited / denom_wall, color="tab:purple", lw=1.6,
        label="deposited / wall")
ax.plot(t_plot, 100.0 * P_net / denom_wall, color="tab:brown", lw=1.6,
        label="net / wall")
ax.axhline(100.0, color="gray", lw=0.8, alpha=0.6)
ax.set_ylim(0, 105)
ax.set_ylabel("% of P_wall")
ax.set_title("Wall efficiency  (% of P_wall, R_comp loss excluded)")
ax.grid(True, alpha=0.3)
ax.legend(fontsize=8, loc="best")
psr._add_phase_lines(ax, phase_events)

# Panel 3: load efficiency -- injected, deposited and net as % of P_load. Both
# the R_comp loss and the wall stage are upstream of the load, so neither appears
# here (P_load = 100%).
ax = axes[2]
denom_load = _pct_denom(P_load)
ax.plot(t_plot, 100.0 * P_injected / denom_load, color="tab:green", lw=1.6,
        label="injected / load")
ax.plot(t_plot, 100.0 * P_deposited / denom_load, color="tab:purple", lw=1.6,
        label="deposited / load")
ax.plot(t_plot, 100.0 * P_net / denom_load, color="tab:brown", lw=1.6,
        label="net / load")
ax.axhline(100.0, color="gray", lw=0.8, alpha=0.6)
ax.set_ylim(0, 105)
ax.set_xlabel(time_label)
ax.set_ylabel("% of P_load")
ax.set_title("Load efficiency  (% of P_load, R_comp + wall excluded)")
ax.grid(True, alpha=0.3)
ax.legend(fontsize=8, loc="best")
psr._add_phase_lines(ax, phase_events)

fig.suptitle("Power efficiency\n" + psr._plot_title(result), fontsize=11)
display(fig)
plt.close(fig)

# Numeric read at the peak of P_wall: the efficiency of each stage at full drive.
k = int(np.argmax(P_wall)) if np.max(P_wall) > 0 else 0
w = P_wall[k] if P_wall[k] > 0 else np.nan
ld = P_load[k] if P_load[k] > 0 else np.nan
unit = time_label.split("[")[-1].rstrip("]").strip()
print(f"at peak P_wall (t = {t_plot[k]:.3g} {unit}):")
print(f"  {'stage':26} {'power [W]':>12} {'% wall':>9} {'% load':>9}")
for label, p in (("P_wall", P_wall[k]), ("P_load", P_load[k]),
                 ("P_comp (R_comp loss)", P_comp[k]), ("P_injected", P_injected[k]),
                 ("P_deposited", P_deposited[k]),
                 ("P_net (inj - e sheath)", P_net[k])):
    print(f"  {label:26} {p:12.4g} {100 * p / w:8.1f} {100 * p / ld:8.1f}")